In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [2]:
# !pip install google-genai

In [3]:
from pathlib import Path
import os
from typing import Dict, List

import duckdb
import pandas as pd
from pymongo import MongoClient
from dotenv import load_dotenv
from google import genai

In [4]:
# load environment variables
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError("CompaniesHouse_API_KEY is not set.")

True

In [5]:
PROJECT_ROOT = Path.cwd()
DUCKDB_PATH = PROJECT_ROOT / "data" / "warehouse" / "project.duckdb"
REPORTS_DIR = PROJECT_ROOT / "artifacts" / "reports"

# Connect DuckDB
duckdb_con = duckdb.connect(str(DUCKDB_PATH))

# Connect MongoDB
mongodb_client = MongoClient("mongodb://localhost:27017/")
mongodb = mongodb_client["uk_entity_review"]
policy_collection = mongodb["policy_documents"]

print("DuckDB connected:", DUCKDB_PATH)
print("MongoDB connected:", mongodb.name)

# Connect Gemini
gemini_client = genai.Client(api_key=GEMINI_API_KEY)

print("Reports directory:", REPORTS_DIR)
print("DuckDB connected:", DUCKDB_PATH)
print("MongoDB connected:", mongodb.name)
print("Gemini key loaded:", GEMINI_API_KEY is not None)

DuckDB connected: /Users/lingzitong/Desktop/MSIN0166 Individual Assignment/data/warehouse/project.duckdb
MongoDB connected: uk_entity_review
Reports directory: /Users/lingzitong/Desktop/MSIN0166 Individual Assignment/artifacts/reports
DuckDB connected: /Users/lingzitong/Desktop/MSIN0166 Individual Assignment/data/warehouse/project.duckdb
MongoDB connected: uk_entity_review
Gemini key loaded: True


In [6]:
def format_entity_brief(entity_record: dict) -> str:
    lines = [
        f"Entity ID: {entity_record.get('entity_id')}",
        f"Entity Name: {entity_record.get('entity_name')}",
        f"Incorporation Date: {str(entity_record.get('incorporation_date'))[:10]}",
        f"Account Category: {entity_record.get('account_category')}",
        f"Primary SIC: {entity_record.get('sic_text_1')}",
        f"Mortgage Charges: {entity_record.get('num_mort_charges')}",
        f"Outstanding Mortgage Charges: {entity_record.get('num_mort_outstanding')}",
        f"Review Priority Score: {entity_record.get('review_priority_score')}",
        f"Review Priority Band: {entity_record.get('review_priority_band')}",
    ]
    return "\n".join(lines)

In [7]:
def get_triggered_signals(entity_record: dict) -> list[str]:
    signal_columns = [
        "new_entity_flag",
        "missing_location_flag",
        "no_accounts_filed_flag",
        "has_outstanding_mortgage_flag",
        "mixed_mortgage_profile_flag",
    ]
    return [signal_name for signal_name in signal_columns if entity_record.get(signal_name) == 1]

In [ ]:
def format_triggered_signals(triggered_signals: List[str]) -> str:
    signal_labels = {
        "new_entity_flag": "New entity signal",
        "missing_location_flag": "Missing location signal",
        "no_accounts_filed_flag": "No accounts filed signal",
        "has_outstanding_mortgage_flag": "Outstanding mortgage signal",
        "mixed_mortgage_profile_flag": "Mixed mortgage profile signal",
    }

    signal_explanations = {
        "new_entity_flag": "The entity is recently incorporated, resulting in limited historical information for performance evaluation.",
        "missing_location_flag": "The registry profile contains incomplete location information, which may weaken basic profile completeness and traceability.",
        "no_accounts_filed_flag": "The registry profile has no filed accounts, which may reduce disclosure visibility and limits routine filing-based review.",
        "has_outstanding_mortgage_flag": "The entity has outstanding registered charges, which may warrant additional attention to structural and financial profile of the entity.",
        "mixed_mortgage_profile_flag": "The entity has a mixed mortgage profile, with outstanding charges alongside satisfied or part-satisfied charge history, which indicates greater structural complexity.",
    }

    lines = []
    
    for signal in triggered_signals:
        label = signal_labels.get(signal, signal)
        explanation = signal_explanations.get(signal, "No explanation available.")
        lines.append(f"- {label}: {explanation}")
    return "\n".join(lines)

In [9]:
def format_enrichment_brief(enrichment_record: Dict) -> str:
    if not enrichment_record:
        return "No external API enrichment available."

    lines = [
        f"API Company Status: {enrichment_record.get('company_status_api')}",
        f"API Date of Creation: {enrichment_record.get('date_of_creation_api')}",
        f"Has Insolvency History: {enrichment_record.get('has_insolvency_history')}",
        f"API Type: {enrichment_record.get('type_api')}",
        f"Jurisdiction: {enrichment_record.get('jurisdiction')}",
    ]
    return "\n".join(lines)

In [10]:
def format_policy_snippets(relevant_documnets: List[Dict], max_characters_per_document: int = 1200) -> str:
    snippets = []
    for document in relevant_documnets:
        snippet_text = document["content"][:max_characters_per_document]
        snippets.append(f"{document['title']}:\n{snippet_text}")
    return "\n\n".join(snippets)

In [11]:
def generate_rule_based_review_note(
    entity_record: Dict,
    enrichment_record: Dict,
    triggered_signals: List[str]
) -> str:
    entity_name = entity_record.get("entity_name", "Unknown Entity")
    priority_band = entity_record.get("review_priority_band", "Unknown")
    review_score = entity_record.get("review_priority_score", "Unknown")

    rationale_line = (
        f"{entity_name} is classified as {priority_band} priority with a review score of {review_score}."
    )

    signal_text = format_triggered_signals(triggered_signals)

    if enrichment_record:
        enrichment_text = (
            f"The Companies House API indicates company status = {enrichment_record.get('company_status_api')}, "
            f"type = {enrichment_record.get('type_api')}, jurisdiction = {enrichment_record.get('jurisdiction')}, and insolvency history = {enrichment_record.get('has_insolvency_history')}."
        )
    else:
        enrichment_text = "No external API enrichment available."

    review_considerations = [
        "- Assess whether the absence of filed accounts is consistent with the incorporation date and the expected statutory filing timeline.",
        "- Conduct a more detailed review of the registered charge profile, with particular attention to the volume and legal status of active charges.",
        "- Assess whether the ownership and control structure requires further verification.",
        "- Consider whether the overall review profile warrants further documentary review or enhanced due diligence."
    ]

    review_note = f"""
Risk Review Support Note — {entity_name}

1. Review Priority Rationale
{rationale_line}

2. Triggered Indicators
{signal_text}

3. External Official Context
{enrichment_text}

4. Further Review Considerations
{chr(10).join(review_considerations)}
    """.strip()

    return review_note

In [12]:
def generate_llm_review_note(review_prompt: str) -> str:
    response = gemini_client.models.generate_content(
        model="gemini-3.1-flash-lite-preview",
        contents=review_prompt,
    )
    return response.text

In [13]:
high_entities_df = duckdb_con.execute("""
    SELECT
        entity_id,
        entity_name,
        incorporation_date,
        account_category,
        num_mort_charges,
        num_mort_outstanding,
        sic_text_1,
        new_entity_flag,
        missing_location_flag,
        no_accounts_filed_flag,
        has_outstanding_mortgage_flag,
        mixed_mortgage_profile_flag,
        review_priority_score,
        review_priority_band
    FROM entity_risk_signals_v2
    WHERE review_priority_band = 'High'
    ORDER BY entity_id
    LIMIT 2
""").fetchdf()

medium_entities_df = duckdb_con.execute("""
    SELECT
        entity_id,
        entity_name,
        incorporation_date,
        account_category,
        num_mort_charges,
        num_mort_outstanding,
        sic_text_1,
        new_entity_flag,
        missing_location_flag,
        no_accounts_filed_flag,
        has_outstanding_mortgage_flag,
        mixed_mortgage_profile_flag,
        review_priority_score,
        review_priority_band
    FROM entity_risk_signals_v2
    WHERE review_priority_band = 'Medium'
    ORDER BY entity_id
    LIMIT 1
""").fetchdf()

sample_entities_df = pd.concat([high_entities_df, medium_entities_df], ignore_index=True)
sample_entities_df

,entity_id,entity_name,incorporation_date,account_category,num_mort_charges,num_mort_outstanding,sic_text_1,new_entity_flag,missing_location_flag,no_accounts_filed_flag,has_outstanding_mortgage_flag,mixed_mortgage_profile_flag,review_priority_score,review_priority_band
0,16332344,BLUBRIGHT PROPERTY LTD,2025-03-21,NO ACCOUNTS FILED,2,1,68209 - Other letting and operating of own or ...,1,0,1,1,1,4,High
1,16333879,OVERBROOK HOLDINGS LIMITED,2025-03-21,NO ACCOUNTS FILED,3,2,41100 - Development of building projects,1,0,1,1,1,4,High
2,00046050,COOPER BROTHERS & SONS LIMITED,1895-11-22,NO ACCOUNTS FILED,2,1,None Supplied,0,0,1,1,1,3,Medium


In [14]:
relevant_documents = list(
    policy_collection.find(
        {
            "title": {
                "$in": [
                    "Risk Flag Guidelines",
                    "Manual Review Checklist",
                    "Entity Review Policy"
                ]
            }
        },
        {
            "_id": 0,
            "doc_id": 1,
            "title": 1,
            "content": 1
        }
    )
)

print("Retrieved policy docs:", [document["title"] for document in relevant_documents])

Retrieved policy docs: ['Entity Review Policy', 'Risk Flag Guidelines', 'Manual Review Checklist']


In [15]:
def build_review_package_for_entity(entity_row: pd.Series) -> Dict:
    entity_record = entity_row.to_dict()
    entity_id = entity_record["entity_id"]

    external_df = duckdb_con.execute(f"""
        SELECT *
        FROM entity_external_enrichment
        WHERE entity_id = '{entity_id}'
    """).fetchdf()

    if not external_df.empty:
        enrichment_record = external_df.iloc[0].to_dict()
    else:
        enrichment_record = {}

    entity_brief = format_entity_brief(entity_record)
    triggered_signals = get_triggered_signals(entity_record)
    triggered_signals_text = format_triggered_signals(triggered_signals)
    enrichment_brief = format_enrichment_brief(enrichment_record)
    policy_text = format_policy_snippets(relevant_documents, max_characters_per_document=1200)

    review_prompt = f"""
You are assisting with a UK business entity risk review support workflow.
Your role is to produce a structured review support note for analyst.

Entity Summary:
{entity_brief}

Triggered Review Indicators:
{triggered_signals_text}

External Official Enrichment:
{enrichment_brief}

Relevant Policy Guidance Snippets:
{policy_text}

Write a concise professional review support note with the following headings:
1. Review priority rationale
2. Triggered indicators
3. External official context
4. Further review considerations

Constraints:
- Do not make final legal, regulatory, or compliance determinations.
- Do not state that misconduct, illegality, or sanctions exposure has been established.
- Keep the tone cautious, analytical, and supportive of human review.
    """.strip()

    return {
        "entity_record": entity_record,
        "enrichment_record": enrichment_record,
        "triggered_signals": triggered_signals,
        "entity_brief": entity_brief,
        "enrichment_brief": enrichment_brief,
        "review_prompt": review_prompt,
    }

In [16]:
sample_review_package = build_review_package_for_entity(sample_entities_df.iloc[0])

print(sample_review_package["entity_brief"])

print("\n--- Triggered Signals ---")
print(format_triggered_signals(sample_review_package["triggered_signals"]))

print("\n--- Enrichment Brief ---")
print(sample_review_package["enrichment_brief"])

print("\n--- Prompt Preview ---")
print(sample_review_package["review_prompt"][:2000])

Entity ID: 16332344
Entity Name: BLUBRIGHT PROPERTY LTD
Incorporation Date: 2025-03-21
Account Category: NO ACCOUNTS FILED
Primary SIC: 68209 - Other letting and operating of own or leased real estate
Mortgage Charges: 2
Outstanding Mortgage Charges: 1
Review Priority Score: 4
Review Priority Band: High

--- Triggered Signals ---
- New entity signal: The entity is recently incorporated, resulting in limited historical information for performance evaluation.
- No accounts filed signal: The registry profile has no filed accounts, which may reduce disclosure visibility and limits routine filing-based review.
- Outstanding mortgage signal: The entity has outstanding registered charges, which may warrant additional attention to structural and financial profile of the entity
- Mixed mortgage profile signal: The entity has a mixed mortgage profile, with outstanding charges alongside satisfied or part-satisfied charge history, which indicates greater structural complexity

--- Enrichment Brief

In [17]:
sample_prompt = sample_review_package["review_prompt"]

try:
    sample_llm_review_note = generate_llm_review_note(sample_prompt)
except Exception as error:
    sample_llm_review_note = None
    print("LLM call failed:", error)

sample_rule_based_review_note = generate_rule_based_review_note(
    entity_record=sample_review_package["entity_record"],
    enrichment_record=sample_review_package["enrichment_record"],
    triggered_signals=sample_review_package["triggered_signals"]
)

print("=== LLM REVIEW NOTE ===")
print(sample_llm_review_note if sample_llm_review_note else "No LLM note generated.")

print("\n\n=== RULE-BASED FALLBACK NOTE ===")
print(sample_rule_based_review_note)

=== LLM REVIEW NOTE ===
### Review Support Note: BLUBRIGHT PROPERTY LTD (ID: 16332344)

**1. Review priority rationale**
This entity is classified as **High Priority** primarily due to its recent incorporation and the immediate presence of secured financial obligations. As a newly formed company (incorporated March 2025), the entity lacks historical financial reporting, which limits the ability to assess its operational performance or financial stability. The presence of outstanding mortgage charges shortly after incorporation warrants further scrutiny to understand the entity’s capital structure and the nature of the secured debt.

**2. Triggered indicators**
*   **New entity signal:** Incorporated within the last 365 days; limited operational track record.
*   **No accounts filed signal:** Consistent with the recent incorporation date but reduces transparency regarding the entity’s initial capitalisation and financial position.
*   **Outstanding mortgage signal:** Indicates the prese

In [18]:
review_outputs = []

for _, row in sample_entities_df.iterrows():
    review_package = build_review_package_for_entity(row)

    try:
        llm_review_note = generate_llm_review_note(review_package["review_prompt"])
        llm_generation_status = "success"
    except Exception as error:
        llm_review_note = None
        llm_generation_status = f"failed: {error}"

    rule_based_review_note = generate_rule_based_review_note(
        entity_record=review_package["entity_record"],
        enrichment_record=review_package["enrichment_record"],
        triggered_signals=review_package["triggered_signals"]
    )

    final_review_note = llm_review_note if llm_review_note else rule_based_review_note

    review_outputs.append({
        "entity_id": review_package["entity_record"]["entity_id"],
        "entity_name": review_package["entity_record"]["entity_name"],
        "review_priority_band": review_package["entity_record"]["review_priority_band"],
        "review_priority_score": review_package["entity_record"]["review_priority_score"],
        "triggered_signals": review_package["triggered_signals"],
        "llm_generation_status": llm_generation_status,
        "review_prompt": review_package["review_prompt"],
        "review_note": final_review_note,
    })

review_outputs_df = pd.DataFrame(review_outputs)
review_outputs_df[["entity_id", "entity_name", "review_priority_band", "review_priority_score", "llm_generation_status"]]

,entity_id,entity_name,review_priority_band,review_priority_score,llm_generation_status
0,16332344,BLUBRIGHT PROPERTY LTD,High,4,success
1,16333879,OVERBROOK HOLDINGS LIMITED,High,4,success
2,00046050,COOPER BROTHERS & SONS LIMITED,Medium,3,success


In [23]:
for output in review_outputs:
    entity_id = output["entity_id"]
    priority_band = str(output["review_priority_band"]).lower()

    note_path = REPORTS_DIR / f"review_note_{priority_band}_{entity_id}.md"
    prompt_path = REPORTS_DIR / f"review_prompt_{priority_band}_{entity_id}.txt"

    note_path.write_text(output["review_note"], encoding="utf-8")
    prompt_path.write_text(output["review_prompt"], encoding="utf-8")

print("Saved review notes and prompts to:", REPORTS_DIR)

2714

5450

2514

5430

2591

5185

Saved review notes and prompts to: /Users/lingzitong/Desktop/MSIN0166 Individual Assignment/artifacts/reports


In [24]:
summary_output_path = REPORTS_DIR / "review_outputs_summary.csv"
review_outputs_df.drop(columns=["review_prompt", "review_note"]).to_csv(summary_output_path, index=False)

print("Saved review summary table to:", summary_output_path)
review_outputs_df

Saved review summary table to: /Users/lingzitong/Desktop/MSIN0166 Individual Assignment/artifacts/reports/review_outputs_summary.csv


,entity_id,entity_name,review_priority_band,review_priority_score,triggered_signals,llm_generation_status,review_prompt,review_note
0,16332344,BLUBRIGHT PROPERTY LTD,High,4,"[new_entity_flag, no_accounts_filed_flag, has_...",success,You are assisting with a UK business entity ri...,### Entity Review Support Note\n\n**Entity Nam...
1,16333879,OVERBROOK HOLDINGS LIMITED,High,4,"[new_entity_flag, no_accounts_filed_flag, has_...",success,You are assisting with a UK business entity ri...,### Entity Risk Review Support Note: OVERBROOK...
2,00046050,COOPER BROTHERS & SONS LIMITED,Medium,3,"[no_accounts_filed_flag, has_outstanding_mortg...",success,You are assisting with a UK business entity ri...,**Entity Risk Review Support Note**\n\n**Entit...


In [25]:
example_note_path = REPORTS_DIR / f"review_note_{str(review_outputs[0]['review_priority_band']).lower()}_{review_outputs[0]['entity_id']}.md"
print(example_note_path.read_text(encoding="utf-8")[:4000])

### Entity Review Support Note

**Entity Name:** BLUBRIGHT PROPERTY LTD  
**Entity ID:** 16332344  
**Review Priority:** High

---

#### 1. Review priority rationale
The entity is classified as **High** priority primarily due to its status as a newly incorporated firm with an absence of filed financial accounts. While the lack of accounts is currently consistent with its recent incorporation date (March 2025), the presence of active mortgage charges suggests an immediate commencement of financial activity. The combination of limited historical performance data and established debt obligations necessitates further scrutiny to establish the entity’s financial and operational profile.

#### 2. Triggered indicators
*   **New Entity Signal:** Incorporated within the last 365 days; limited operational history available.
*   **No Accounts Filed Signal:** Reduced transparency regarding financial position; expected for a new entity but limits standard filing-based verification.
*   **Outstandin

In [26]:
duckdb_con.close()
print("DuckDB connection closed.")

DuckDB connection closed.
